<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/cross_dataset_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing feature groups across datasets

In our earlier notebooks we found that on Sparkov the useful part of velocity is the time gap, while on TabFormer most of the signal comes from where the merchant is. These results came from separate notebooks, and the Credit Card Fraud 2026 dataset was only checked with a quick split and one seed. In this notebook we run one protocol on all three datasets and compare the same seven feature groups, so the results can be put side by side in one table.

| Feature group | What it contains |
|---|---|
| Distance only | distance from the card's previous transaction |
| Time gap only | hours since the card's previous transaction (`gap_log`) |
| Velocity only | distance divided by the time gap |
| Location only | where the merchant is (see the second table) |
| Our six features | `amt_log`, `category_risk`, `is_night`, `gap_log`, `age`, `gender_flag` |
| Six + location | our six features and the location features |
| All native columns | every usable column the dataset provides; this measures how much signal the dataset has and is not part of our method |

The protocol is the same as in our Sparkov notebooks:
1. We split by time. The newest part is the test set, and the last 20% of the training period is validation.
2. `category_risk` and every text-column encoding are fitted on the training part only.
3. We use the Random Forest Tuned settings (200 trees, `max_features=3`, `min_samples_leaf=5`) with five seeds.
4. The alert threshold is the best F2 on validation, applied once to the test set.
5. The 95% intervals come from a day-block bootstrap with 500 resamples of whole test days.

We also report the test fraud rate (prevalence) and the **lift**, which is the PR-AUC divided by the prevalence. A model that ranks transactions at random has a PR-AUC equal to the fraud rate, so a PR-AUC of 0.02 means very different things on a dataset with 0.08% fraud and one with 2% fraud. A lift of 1 means no better than chance.

**Distance and location features in each dataset**

| Dataset | Distance | Location features |
|---|---|---|
| Sparkov | km between the merchant coordinates of consecutive transactions | distance from home (km), distance from the previous transaction (km) |
| TabFormer | TabFormer has no merchant coordinates, so we use a proxy: 1 if the merchant city changed since the card's previous transaction | merchant outside the home state, merchant abroad (as in `tabformer_validation.ipynb`) |
| CC Fraud 2026 | the dataset's `distance_from_last_tx_km` | distance from home, distance from the last transaction, international, high-risk country, fraud rate of the merchant country |

We run the notebook once per dataset by changing `DATASET` in cell 1.1. Each run saves its results to Google Drive, and Section 7 combines the finished datasets into one table.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook. `QUICK = True` gives a fast preview with one seed, 100 trees and 200 bootstrap resamples; the results we report use `QUICK = False`.

In [1]:
# 1.1 Settings: we change DATASET and run the notebook once per dataset
DATASET = 'cc2026'          # 'sparkov', 'tabformer' or 'cc2026'
QUICK   = False             # True = 1 seed, 100 trees, 200 bootstrap resamples (a fast preview only)

SEEDS     = [42] if QUICK else [42, 43, 44, 45, 46]
N_TREES   = 100 if QUICK else 200
N_BOOT    = 200 if QUICK else 500
BOOT_SEED = 2026

# TabFormer: the same user sample and window as tabformer_validation.ipynb
TF_N_USERS, TF_SAMPLE_SEED, TF_WINDOW_START = 400, 42, '2015-01-01'

KAGGLE_SLUGS = {
    'sparkov':   'kartik2112/fraud-detection',
    'tabformer': 'ealtman2019/credit-card-transactions',
    'cc2026':    'mohankrishnathalla/credit-card-fraud-detection-dataset-2026',
}
assert DATASET in KAGGLE_SLUGS, f'unknown DATASET {DATASET!r}'
print(f'dataset: {DATASET} | seeds: {SEEDS} | trees: {N_TREES} | bootstrap: {N_BOOT}')

dataset: cc2026 | seeds: [42, 43, 44, 45, 46] | trees: 200 | bootstrap: 500


In [2]:
# 1.2 Download the dataset from Kaggle
import os, glob
from getpass import getpass

DATA_DIR = f'/content/data/{DATASET}'
if not glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True):
    os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
    os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')
    !kaggle datasets download -d {KAGGLE_SLUGS[DATASET]} -p {DATA_DIR} --unzip
for f in sorted(glob.glob(f'{DATA_DIR}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mohankrishnathalla/credit-card-fraud-detection-dataset-2026
License(s): CC0-1.0
100% 6.65M/6.65M [00:00<00:00, 84.6MB/s]

     25.7 MB  /content/data/cc2026/credit_card_fraud_detection_2026.csv


In [3]:
# 1.3 Clone the project repository to use our helpers
REPO = 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'
if not os.path.exists('src/evaluation.py'):
    if not os.path.exists(REPO):
        github_token = getpass('Paste your GitHub token: ').strip()
        !git clone https://{github_token}@github.com/subbareddypullalacheruvu/{REPO}.git
    %cd {REPO}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 325, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 325 (delta 114), reused 53 (delta 44), pack-reused 149 (from 1)
Receiving objects: 100% (325/325), 11.73 MiB | 15.14 MiB/s, done.
Resolving deltas: 100% (192/192), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Imports and the Google Drive folders for progress and results
import sys
import numpy as np
from IPython.display import display
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk, haversine_distance
from src.evaluation import chronological_val_split, run_seeds, bootstrap_model, summarise, format_ci
pd.set_option('display.max_columns', 60)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol'
CHECKPOINT_DIR = f"{BASE}/{DATASET}{'_quick' if QUICK else ''}"
RESULTS_DIR = f'{BASE}/results'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('progress ->', CHECKPOINT_DIR)
print('results  ->', RESULTS_DIR)

Mounted at /content/drive
progress -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/cc2026
results  -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results


## 2. Loading the data
Each dataset stores its columns differently, so we write one loader per dataset. Every loader returns the same helper columns (`_time`, `_y` for the label, `_card`, `_amount`, `_category`, `age` and `gender_flag`) next to the raw columns, so the rest of the notebook is the same for all datasets. It also returns the dataset's own columns for the all-native-columns run (`NATIVE_COLS`) and the columns that hold codes or names rather than quantities (`TEXT_COLS`).

In [5]:
# 2.1 One loader per dataset
def csv_files():
    return glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True)

def load_sparkov():
    files = csv_files()
    tr = pd.read_csv(next(f for f in files if f.endswith('fraudTrain.csv')))
    te = pd.read_csv(next(f for f in files if f.endswith('fraudTest.csv')))
    tr['_split'], te['_split'] = 'train', 'test'          # Kaggle's own train/test files, as in all our notebooks
    d = pd.concat([tr, te], ignore_index=True)
    d = d.drop(columns=[c for c in d.columns if c.startswith('Unnamed')])
    d['_time'] = pd.to_datetime(d['trans_date_trans_time'])
    d['_y'] = d['is_fraud'].astype(int)
    d['_card'] = d['cc_num'].astype(str)
    d['_amount'] = d['amt'].astype(float)
    d['_category'] = d['category'].astype(str)
    d['age'] = (d['_time'] - pd.to_datetime(d['dob'])).dt.days / 365.25
    d['gender_flag'] = (d['gender'] == 'M').astype(int)
    skip = {'trans_date_trans_time', 'cc_num', 'first', 'last', 'street', 'trans_num', 'unix_time',
            'dob', 'is_fraud', '_split'}
    native = [c for c in tr.columns if c not in skip and not c.startswith('Unnamed')]
    text = {'category', 'gender', 'city', 'state', 'job', 'merchant', 'zip'}
    return d, native, text

def load_tabformer():
    files = csv_files()
    users_csv = next(f for f in files if 'user' in os.path.basename(f).lower()
                     and 'transaction' not in os.path.basename(f).lower())
    txn_csv = max((f for f in files if 'transaction' in os.path.basename(f).lower()), key=os.path.getsize)
    users = pd.read_csv(users_csv)
    users['User'] = np.arange(len(users))               # transactions refer to users by row number
    rng = np.random.default_rng(TF_SAMPLE_SEED)
    sampled = set(rng.choice(users['User'].values, size=min(TF_N_USERS, len(users)), replace=False).tolist())
    parts = [ch[ch['User'].isin(sampled)] for ch in pd.read_csv(txn_csv, chunksize=2_000_000, low_memory=False)]
    raw = pd.concat(parts, ignore_index=True)
    txn_cols = list(raw.columns)

    d = raw.copy()
    d['_time'] = pd.to_datetime(dict(year=d['Year'], month=d['Month'], day=d['Day']), errors='coerce') \
                 + pd.to_timedelta(d['Time'].astype(str) + ':00', errors='coerce')
    d['_y'] = d['Is Fraud?'].astype(str).str.strip().str.lower().isin(['yes', '1', 'true']).astype(int)
    d['_amount'] = pd.to_numeric(d['Amount'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
    d['_card'] = d['User'].astype(str) + '_' + d['Card'].astype(str)     # one card = user + card number
    d['_category'] = d['MCC'].astype(str)
    user_cols = [c for c in users.columns if c not in d.columns]
    d = d.merge(users[['User'] + user_cols], on='User', how='left')
    birth = pd.to_datetime(dict(year=d['Birth Year'], month=d['Birth Month'], day=1), errors='coerce')
    d['age'] = (d['_time'] - birth).dt.days / 365.25
    d['gender_flag'] = d['Gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])

    skip = {'User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Is Fraud?', 'Merchant Name', 'Zip',
            'Person', 'Address', 'Apartment', 'Zipcode', 'Birth Month'}
    native = [c for c in txn_cols + user_cols if c not in skip]
    text = {'MCC', 'Merchant State', 'Merchant City', 'State', 'Gender', 'City', 'Use Chip', 'Errors?'}
    return d, native, text

def load_cc2026():
    d = pd.read_csv(max(csv_files(), key=os.path.getsize), low_memory=False)
    raw_cols = list(d.columns)
    d['_time'] = pd.to_datetime(d['transaction_datetime'], errors='coerce')
    d['_y'] = pd.to_numeric(d['is_fraud'], errors='coerce').fillna(0).astype(int)
    d['_card'] = d['card_id'].astype(str)
    d['_amount'] = d['transaction_amount'].astype(float)
    d['_category'] = d['merchant_category'].astype(str)
    d['age'] = pd.to_numeric(d['customer_age'], errors='coerce')
    d['gender_flag'] = d['customer_gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])
    # identifiers, the time stamp, the label and the two columns that describe the fraud itself
    skip = {'transaction_id', 'customer_id', 'card_id', 'merchant_id', 'transaction_datetime',
            'is_fraud', 'fraud_type', 'fraud_amount_lost'}
    native = [c for c in raw_cols if c not in skip]
    text = {'merchant_category', 'merchant_country', 'transaction_day_of_week', 'customer_gender',
            'customer_state', 'transaction_channel', 'device_type', 'browser', 'email_domain'}
    return d, native, text

d, NATIVE_COLS, TEXT_COLS = {'sparkov': load_sparkov, 'tabformer': load_tabformer, 'cc2026': load_cc2026}[DATASET]()
print(f'{len(d):,} rows, {d["_card"].nunique():,} cards, fraud rate {d["_y"].mean():.4%}')
print(f'{d["_time"].min()} to {d["_time"].max()}')
print(f'{len(NATIVE_COLS)} native columns:', NATIVE_COLS)

100,000 rows, 8,000 cards, fraud rate 2.0550%
2026-01-01 02:28:51 to 2026-12-31 23:45:47
35 native columns: ['transaction_amount', 'merchant_category', 'merchant_country', 'transaction_hour', 'transaction_day_of_week', 'card_present', 'customer_age', 'customer_gender', 'customer_state', 'account_age_days', 'credit_limit', 'credit_score', 'num_cards_held', 'transactions_last_1h', 'transactions_last_24h', 'transactions_last_7d', 'amount_last_1h', 'amount_last_24h', 'days_since_last_transaction', 'distance_from_home_km', 'distance_from_last_tx_km', 'international_transaction', 'high_risk_country', 'amount_vs_avg_ratio', 'used_chip', 'used_pin', 'transaction_channel', 'device_type', 'browser', 'is_weekend', 'avg_transaction_amount_30d', 'std_transaction_amount_30d', 'num_declined_last_30d', 'num_chargebacks_ever', 'email_domain']


## 3. Building the features
The time gap, distance and velocity only look back at the same card's previous transaction, so we calculate them on the full history before splitting, as in our other notebooks.

The time gap is calculated per card (`cc_num` in Sparkov, `User` and `Card` together in TabFormer, `card_id` in CC 2026), after sorting by the full date and time, and it is in hours on every dataset. TabFormer only records the time to the minute, so its gaps are rounded to the minute. A card's first transaction has no previous one, so its gap, distance and velocity are set to -1.

The table at the end compares the median time gap and distance of fraud and normal transactions.

In [6]:
# 3.1 Time gap, distance, velocity and location features
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]

d = d.sort_values(['_card', '_time']).reset_index(drop=True)
by_card = d.groupby('_card')
d['_gap_h'] = by_card['_time'].diff().dt.total_seconds() / 3600

if DATASET == 'sparkov':
    prev_lat, prev_lon = by_card['merch_lat'].shift(1), by_card['merch_long'].shift(1)
    d['_dist'] = haversine_distance(d['merch_lat'], d['merch_long'], prev_lat, prev_lon)
    d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 3600)          # +1 second so we never divide by zero
    d['dist_home_log'] = np.log1p(haversine_distance(d['lat'], d['long'], d['merch_lat'], d['merch_long']))
    LOCATION = ['dist_home_log', 'dist_log']

elif DATASET == 'tabformer':
    # no GPS points: distance proxy = 1 if the merchant city changed since the card's previous transaction
    city = d['Merchant City'].astype(str)
    prev_city = by_card['Merchant City'].shift(1)
    d['_dist'] = np.where(prev_city.isna(), np.nan, (city != prev_city.astype(str)).astype(float))
    d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 60)            # +1 minute (TabFormer's clock resolution)
    ms, has = d['Merchant State'], d['Merchant State'].notna()
    d['abroad'] = (has & (ms.astype(str).str.len() > 2)).astype(int)          # foreign merchants have a country name
    d['outside_home_state'] = (has & (ms.astype(str) != d['State'].astype(str))).astype(int)
    LOCATION = ['outside_home_state', 'abroad']
    d = d[d['_time'] >= TF_WINDOW_START].copy()                # gaps were computed on the full history first

elif DATASET == 'cc2026':
    d['_dist'] = pd.to_numeric(d['distance_from_last_tx_km'], errors='coerce')
    d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 3600)
    d['dist_home_log'] = np.log1p(pd.to_numeric(d['distance_from_home_km'], errors='coerce').clip(lower=0))
    LOCATION = ['dist_home_log', 'dist_log', 'international_transaction', 'high_risk_country', 'merchant_country_risk']

d['gap_log'] = np.log1p(d['_gap_h']).fillna(-1)                # -1 = the card's first transaction
d['dist_log'] = np.log1p(d['_dist'].clip(lower=0)).fillna(-1)
d['velocity_log'] = np.log1p(d['_vel'].clip(lower=0)).fillna(-1)
d['amt_log'] = np.log1p(d['_amount'].clip(lower=0)).fillna(0)
d['is_night'] = d['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
d['age'] = d['age'].fillna(-1)
d['day'] = d['_time'].dt.floor('D')
d = d.sort_values('_time').reset_index(drop=True)

check = pd.DataFrame({'fraud': d.loc[d['_y'] == 1, ['_gap_h', '_dist']].median(),
                      'normal': d.loc[d['_y'] == 0, ['_gap_h', '_dist']].median()})
check.index = ['median time gap (hours)', 'median distance' + (' (city changed share)' if DATASET == 'tabformer' else ' (km)')]
check

,fraud,normal
median time gap (hours),444.14,450.027639
median distance (km),16.40,13.300000


## 4. Splitting by time
For Sparkov we keep Kaggle's own train and test files, as in all our Sparkov notebooks. For TabFormer and CC 2026, the last 20% of the period is the test set. In every dataset, the last 20% of the training period is validation.

In [7]:
# 4.1 Train / validation / test
if DATASET == 'sparkov':
    train, test = d[d['_split'] == 'train'], d[d['_split'] == 'test'].copy()
else:
    cutoff_test = d['_time'].min() + 0.8 * (d['_time'].max() - d['_time'].min())
    train, test = d[d['_time'] < cutoff_test], d[d['_time'] >= cutoff_test].copy()
train_sub, val, cutoff_val = chronological_val_split(train, '_time', val_frac=0.2)
del train

for name, part in [('train_sub', train_sub), ('val', val), ('test', test)]:
    print(f"{name:9s} {part['_time'].min()} to {part['_time'].max()}  rows={len(part):>9,}  "
          f"fraud={part['_y'].sum():>6,}  rate={part['_y'].mean():.4%}")
PREVALENCE = test['_y'].mean()

train_sub 2026-01-01 02:28:51 to 2026-08-22 14:57:50  rows=   63,796  fraud= 1,293  rate=2.0268%
val       2026-08-22 15:08:05 to 2026-10-20 00:08:49  rows=   16,186  fraud=   344  rate=2.1253%
test      2026-10-20 00:24:28 to 2026-12-31 23:45:47  rows=   20,018  fraud=   418  rate=2.0881%


## 5. Encodings fitted on the training part only
`category_risk` is the fraud rate of each merchant category on the training part. For CC 2026 we build `merchant_country_risk` in the same way. For the all-native-columns run, we replace every text column with the fraud rate of each of its values on the training part and keep numeric columns as they are, filling missing values with the training median. We leave out text columns with more than 1,000 values, because they are identifiers rather than features.

In [8]:
# 5.1 category_risk and the native-column encodings
parts = (train_sub, val, test)
risk_map = fit_category_risk(train_sub, '_category', '_y')
for p in parts:
    p['category_risk'] = apply_category_risk(p, '_category', risk_map)

if DATASET == 'cc2026':
    country_map = fit_category_risk(train_sub, 'merchant_country', '_y')
    for p in parts:
        p['merchant_country_risk'] = apply_category_risk(p, 'merchant_country', country_map)

def as_number(s):
    return pd.to_numeric(s.astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')

NATIVE_FEATURES, native_info = [], []
y_fit, prior = train_sub['_y'], train_sub['_y'].mean()
for col in NATIVE_COLS:
    name, s = f'raw__{col}', train_sub[col]
    if col not in TEXT_COLS and pd.api.types.is_bool_dtype(s):
        for p in parts: p[name] = p[col].astype(float)
        used = 'numeric'
    elif col not in TEXT_COLS and pd.api.types.is_numeric_dtype(s):
        fill = s.median()
        for p in parts: p[name] = p[col].astype(float).fillna(fill)
        used = 'numeric'
    elif col not in TEXT_COLS and as_number(s).notna().mean() > 0.95:       # e.g. '$1,234'
        fill = as_number(s).median()
        for p in parts: p[name] = as_number(p[col]).fillna(fill)
        used = 'numeric (parsed text)'
    elif s.astype(str).nunique() <= 1000:
        rates = y_fit.groupby(s.astype(str)).mean()
        for p in parts: p[name] = p[col].astype(str).map(rates).fillna(prior).astype(float)
        used = 'fraud rate of each value'
    else:
        native_info.append({'column': col, 'used as': 'left out (more than 1,000 values)'})
        continue
    NATIVE_FEATURES.append(name)
    native_info.append({'column': col, 'used as': used})
print(f'all-native run uses {len(NATIVE_FEATURES)} columns')
pd.DataFrame(native_info)

all-native run uses 35 columns


,column,used as
0,transaction_amount,numeric
1,merchant_category,fraud rate of each value
2,merchant_country,fraud rate of each value
3,transaction_hour,numeric
4,transaction_day_of_week,fraud rate of each value
5,card_present,numeric
6,customer_age,numeric
7,customer_gender,fraud rate of each value
8,customer_state,fraud rate of each value
9,account_age_days,numeric


## 6. Training the seven feature groups
We train each group with the Random Forest Tuned settings and five seeds. Finished seeds are saved to Google Drive, so if Colab disconnects, re-running skips them. We then bootstrap the test days and report PR-AUC, lift, F2 and MCC with 95% intervals.

In [9]:
# 6.1 Feature groups and training (five seeds each, threshold chosen on validation)
SIX = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']
GROUPS = {
    'Distance only':      ['dist_log'],
    'Time gap only':      ['gap_log'],
    'Velocity only':      ['velocity_log'],
    'Location only':      LOCATION,
    'Our six features':   SIX,
    'Six + location':     SIX + [c for c in LOCATION if c not in SIX],
    'All native columns': NATIVE_FEATURES,
}
for g, cols in GROUPS.items():
    print(f'{g:20s} {len(cols):2d} features' + (f': {cols}' if g != 'All native columns' else ''))

def make_rf(n_features):
    return lambda s: RandomForestClassifier(n_estimators=N_TREES, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
runs = {}
for g, cols in GROUPS.items():
    runs[g] = run_seeds(f'{DATASET} {g}', make_rf(len(cols)),
                        train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                        checkpoint_dir=CHECKPOINT_DIR)

Distance only         1 features: ['dist_log']
Time gap only         1 features: ['gap_log']
Velocity only         1 features: ['velocity_log']
Location only         5 features: ['dist_home_log', 'dist_log', 'international_transaction', 'high_risk_country', 'merchant_country_risk']
Our six features      6 features: ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']
Six + location       11 features: ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag', 'dist_home_log', 'dist_log', 'international_transaction', 'high_risk_country', 'merchant_country_risk']
All native columns   35 features
cc2026 Distance only seed 42: loaded from checkpoint
cc2026 Distance only seed 43: loaded from checkpoint
cc2026 Distance only seed 44: loaded from checkpoint
cc2026 Distance only seed 45: loaded from checkpoint
cc2026 Distance only seed 46: loaded from checkpoint
cc2026 Time gap only seed 42: loaded from checkpoint
cc2026 Time gap only seed 43: loaded from chec

In [10]:
# 6.2 Day-block bootstrap, prevalence and lift
test_days = test['day'].values
rows = []
for g, seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boot = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    s = summarise(y_test, pairs, boot)
    rows.append({'Dataset': DATASET, 'Feature group': g, 'Features': len(GROUPS[g]),
                 'Seeds': len(SEEDS), 'Test rows': len(y_test), 'Test fraud': int(y_test.sum()),
                 'Prevalence': PREVALENCE,
                 'Val PR-AUC': np.mean([r['val_pr_auc'] for r in seed_runs]),
                 'PR-AUC': s['PR-AUC'], 'PR-AUC CI low': s['PR-AUC CI low'], 'PR-AUC CI high': s['PR-AUC CI high'],
                 'Lift': s['PR-AUC'] / PREVALENCE,
                 'Lift CI low': s['PR-AUC CI low'] / PREVALENCE, 'Lift CI high': s['PR-AUC CI high'] / PREVALENCE,
                 'F2': s['F2'], 'F2 CI low': s['F2 CI low'], 'F2 CI high': s['F2 CI high'],
                 'MCC': s['MCC'], 'MCC CI low': s['MCC CI low'], 'MCC CI high': s['MCC CI high'],
                 'Thresholds': s['Thresholds']})
    print(f'{g}: done')

result = pd.DataFrame(rows)
out = f"{RESULTS_DIR}/point1_{DATASET}{'_quick' if QUICK else ''}.csv"
result.to_csv(out, index=False)
os.makedirs('results', exist_ok=True)
result.to_csv(f"results/point1_{DATASET}{'_quick' if QUICK else ''}.csv", index=False)
print('saved ->', out)

show = pd.DataFrame({
    'PR-AUC [95% CI]': [f"{r['PR-AUC']:.4f} [{r['PR-AUC CI low']:.4f}, {r['PR-AUC CI high']:.4f}]" for _, r in result.iterrows()],
    'Lift [95% CI]':   [f"{r['Lift']:.1f} [{r['Lift CI low']:.1f}, {r['Lift CI high']:.1f}]" for _, r in result.iterrows()],
    'F2 [95% CI]':     [f"{r['F2']:.4f} [{r['F2 CI low']:.4f}, {r['F2 CI high']:.4f}]" for _, r in result.iterrows()],
    'MCC [95% CI]':    [f"{r['MCC']:.4f} [{r['MCC CI low']:.4f}, {r['MCC CI high']:.4f}]" for _, r in result.iterrows()],
}, index=result['Feature group'])
print(f'{DATASET}: test fraud rate {PREVALENCE:.4%} ({int(y_test.sum()):,} fraud of {len(y_test):,})')
show

Distance only: done
Time gap only: done
Velocity only: done
Location only: done
Our six features: done
Six + location: done
All native columns: done
saved -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results/point1_cc2026.csv
cc2026: test fraud rate 2.0881% (418 fraud of 20,018)


,PR-AUC [95% CI],Lift [95% CI],F2 [95% CI],MCC [95% CI]
Feature group,,,,
Distance only,"0.0211 [0.0187, 0.0251]","1.0 [0.9, 1.2]","0.0870 [0.0779, 0.0967]","-0.0132 [-0.0269, 0.0012]"
Time gap only,"0.0218 [0.0197, 0.0253]","1.0 [0.9, 1.2]","0.0854 [0.0717, 0.0983]","0.0090 [-0.0038, 0.0236]"
Velocity only,"0.0220 [0.0199, 0.0255]","1.1 [1.0, 1.2]","0.0850 [0.0720, 0.1011]","0.0067 [-0.0064, 0.0220]"
Location only,"0.0503 [0.0425, 0.0635]","2.4 [2.0, 3.0]","0.1631 [0.1435, 0.1827]","0.0880 [0.0713, 0.1043]"
Our six features,"0.0242 [0.0217, 0.0280]","1.2 [1.0, 1.3]","0.1032 [0.0938, 0.1130]","0.0231 [0.0117, 0.0338]"
Six + location,"0.0658 [0.0579, 0.0792]","3.1 [2.8, 3.8]","0.1828 [0.1574, 0.2056]","0.1144 [0.0939, 0.1332]"
All native columns,"0.1425 [0.1226, 0.1707]","6.8 [5.9, 8.2]","0.2876 [0.2627, 0.3123]","0.1977 [0.1788, 0.2177]"


## 7. Combining the datasets
We read the saved results of every finished dataset and build two tables with datasets as rows and feature groups as columns. Table 1a gives the test PR-AUC with its 95% interval, and Table 1b gives the lift. The lift interval is the PR-AUC interval divided by the test fraud rate. If a dataset has both a quick and a full run, we keep the full run.

In [11]:
# 7.1 Table 1: every finished dataset together
ORDER_G = list(GROUPS)
ORDER_D = ['sparkov', 'tabformer', 'cc2026']
NAMES = {'sparkov': 'Sparkov', 'tabformer': 'TabFormer', 'cc2026': 'CC Fraud 2026'}

files = sorted(glob.glob(f'{RESULTS_DIR}/point1_*.csv'))
print('found:', [os.path.basename(f) for f in files])
allres = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# if a dataset has both a quick and a full run, keep the full (five-seed) run
allres = (allres.sort_values('Seeds', ascending=False)
                .drop_duplicates(['Dataset', 'Feature group']).reset_index(drop=True))
allres['Dataset'] = pd.Categorical(allres['Dataset'], [x for x in ORDER_D if x in set(allres['Dataset'])], ordered=True)

def cell(r, m, digits):
    return f"{r[m]:.{digits}f} [{r[m + ' CI low']:.{digits}f}, {r[m + ' CI high']:.{digits}f}]"

def pivot(metric, digits):
    t = (allres.assign(v=[cell(r, metric, digits) for _, r in allres.iterrows()])
               .pivot(index='Dataset', columns='Feature group', values='v')
               .reindex(columns=ORDER_G))
    t.index = [NAMES[x] for x in t.index]
    return t

info = (allres.groupby('Dataset', observed=True)
              .agg(**{'Test rows': ('Test rows', 'first'), 'Test fraud': ('Test fraud', 'first'),
                      'Prevalence %': ('Prevalence', lambda s: round(s.iloc[0] * 100, 3)), 'Seeds': ('Seeds', 'first')}))
info.index = [NAMES[x] for x in info.index]

table1_pr, table1_lift = pivot('PR-AUC', 3), pivot('Lift', 1)
table1_pr.to_csv(f'{RESULTS_DIR}/table1_pr_auc.csv')
table1_lift.to_csv(f'{RESULTS_DIR}/table1_lift.csv')
info.to_csv(f'{RESULTS_DIR}/table1_test_sets.csv')
print('Test sets'); display(info)
print('Table 1a: test PR-AUC [95% CI]'); display(table1_pr)
print('Table 1b: lift = PR-AUC / test fraud rate [95% CI]'); display(table1_lift)

found: ['point1_cc2026.csv']
Test sets


,Test rows,Test fraud,Prevalence %,Seeds
CC Fraud 2026,20018,418,2.088,5


Table 1a: test PR-AUC [95% CI]


Feature group,Distance only,Time gap only,Velocity only,Location only,Our six features,Six + location,All native columns
CC Fraud 2026,"0.021 [0.019, 0.025]","0.022 [0.020, 0.025]","0.022 [0.020, 0.026]","0.050 [0.043, 0.064]","0.024 [0.022, 0.028]","0.066 [0.058, 0.079]","0.143 [0.123, 0.171]"


Table 1b: lift = PR-AUC / test fraud rate [95% CI]


Feature group,Distance only,Time gap only,Velocity only,Location only,Our six features,Six + location,All native columns
CC Fraud 2026,"1.0 [0.9, 1.2]","1.0 [0.9, 1.2]","1.1 [1.0, 1.2]","2.4 [2.0, 3.0]","1.2 [1.0, 1.3]","3.1 [2.8, 3.8]","6.8 [5.9, 8.2]"


## 8. Discussion

### Credit Card Fraud 2026

**The test set.** The data covers 100,000 transactions on 8,000 cards during 2026, with 2.06% fraud. The test set starts on 20 October 2026 and has 20,018 transactions, of which 418 are fraud (2.09%). A model that ranks at random would therefore score a PR-AUC of about 0.021.

**The movement features carry no signal.** Distance only, time gap only and velocity only all score a PR-AUC between 0.021 and 0.022, a lift of 1.0 to 1.1, and every lift interval includes 1. Their MCC intervals also include zero. The medians explain this: fraud and normal transactions have almost the same time gap (444 h against 450 h) and a similar distance from the previous transaction (16.4 km against 13.3 km). Cards in this dataset are used rarely (a median of 12 transactions in the year, about 18 days apart), and fraud does not come in bursts, so a short gap cannot mark a fraud transaction as it does in Sparkov.

**Our six features are close to chance.** They score 0.024 [0.022, 0.028], a lift of only 1.2 [1.0, 1.3]. This confirms the quick check in `external_dataset_check.ipynb` (0.025), now with five seeds, a validation threshold and bootstrap intervals.

**Location is the only engineered signal.** Location only scores 0.050 [0.043, 0.064], a lift of 2.4 [2.0, 3.0], which is the only group of our features that is clearly above chance on its own. Adding the location features to our six features raises the PR-AUC from 0.024 to 0.066 [0.058, 0.079], almost three times higher, and the two intervals do not overlap. F2 rises from 0.103 to 0.183 and MCC from 0.023 to 0.114.

**The dataset's own columns go further, but the task stays hard.** All 35 native columns score 0.143 [0.123, 0.171], a lift of 6.8 [5.9, 8.2], with F2 0.288 and MCC 0.198. The gap between this and our six + location features comes from columns we do not build, such as the channel, the device and whether the card was present, which matches what we saw in the single-column check. Even this upper bound is far below what our six features reach on Sparkov (0.844) and what the native columns reach on TabFormer (0.891), so the fraud labels in this dataset are only weakly related to any of its columns.

**What this means.** On CC 2026 the signal is spatial and contextual, not temporal: the time gap adds nothing, and the location features carry most of what our features can find. This places CC 2026 with TabFormer rather than with Sparkov. The origin of this dataset is not documented in detail, so we treat it as a secondary dataset.